© 2026 by Tamás Takács is licensed under CC BY-NC-SA 4.0. To view a copy of this license, visit https://creativecommons.org/licenses/by-nc-sa/4.0/

English translation managed by Tamás Takács. The translation was produced with AI assistance.

# **International AI Olympiad Online Round (2025) - Starter Notebook**

This **Colab Notebook** was created for the online round of the *2025 International AI Olympiad*, and provides a basic starting point for contestants.  

The notebook covers loading the data and basic visualization, as well as a simple baseline model based on **logistic regression**. This model tries to predict whether a student will be admitted to university or not, based on the **"School Prestige"** variable.  

During the competition you may use any package or framework, as long as the submitted solution complies with the rules set out on the **Kaggle competition page**.  

Both the data provided for the competition and the task itself are **entirely synthetic**, so there is no point in looking for data from external sources.  


# **0. Loading the Required Packages**

In [ ]:
import folium
import json
import torch
import gdown
import numpy as np
import geopandas as gpd
import pandas as pd
from sklearn.linear_model import LogisticRegression

device = "cuda" if torch.cuda.is_available() else "cpu"

# **1. Loading the Data**

Three files are available to you during the competition:

- `train.csv` - Contains the data needed for training, including every feature belonging to each applicant and the target variable (`Felvételi Eredmény`), which indicates whether the given student was admitted to the university (1 - admitted, 0 - not admitted). Each row is identified by a unique `ID`.

- `test.csv` - Contains the test data used for evaluating the competition, **without** the target variable (`Felvételi Eredmény`). Your goal is to produce predictions for this set. In the prediction, every row has an `ID` that links it back to the observation.

- `counties.geojson` - A file in GeoJSON format containing the **geographic boundaries and centroid coordinates** of Hungary's counties. This gives you the opportunity to enrich the applicants' places of residence (at county level) with geospatial data, for example by computing the distance between the place of residence and the university.

---

Your task is to **train a machine learning model** based on the `train.csv` dataset and the map data in `counties.geojson`, which is able to estimate whether the students in the `test.csv` file would be admitted to the university.

The model must return a **probability value between 0 and 1** for every `ID` in `test.csv`. According to the system, this value expresses the probability that the given student gains admission (1 - certain admission, 0 - certain rejection).

## **1.1. Geographic Data**

To solve the task, a `.geojson` file is also available, which contains the **boundaries and geographic coordinates** of Hungary's counties (+ Budapest, whose legal status is capital city and not county) in the **WGS84 (World Geodetic System 1984) coordinate system**. The file describes each county with **latitude and longitude** values. [GeoJSON](https://geojson.org/)

In [ ]:
with open('/kaggle/input/magyar-mi-diakolimpia-online-valogato-2025/counties.geojson', 'r', encoding='utf-8') as f:
    geojson_data = json.load(f)

print(f"Number of Counties + Budapest: {len(geojson_data['features'])}")

## **1.2 Inspecting the structure of the capital, Budapest**

In [ ]:
budapest_data = geojson_data['features'][0]
print(budapest_data.keys())

print(f'Data belonging to Budapest: {budapest_data}')
print(f'Polygon describing the capital: {budapest_data["geometry"]}')
print(f'Properties describing the capital: {budapest_data["properties"]}')

In [ ]:
print(f'First coordinate of the polygon describing the capital: {budapest_data["geometry"]["coordinates"][0][0]}')
print(f'Second coordinate of the polygon describing the capital: {budapest_data["geometry"]["coordinates"][0][1]}')
print(f'Name of the capital: {budapest_data["properties"]["megye"]}')

## **1.3 Displaying the GeoJSON file using *folium***

In [ ]:
gdf = gpd.GeoDataFrame.from_features(geojson_data["features"])

m = folium.Map(location=[47.0, 19.5], zoom_start=7)
for _, row in gdf.iterrows():
    folium.GeoJson(row["geometry"]).add_to(m)

uni_coords = (47.472113, 19.062236)

folium.Marker(
    location=uni_coords,
    popup="University (Budapest)",
    icon=folium.Icon(color="red", icon="graduation-cap", prefix="fa"),
).add_to(m)

m

## **1.4 Distance from the University**

One of your cartographer friends — who, by the way, used to work on the university's administration system — told you over a coffee that **behind the admission decisions** they do indeed **take into account the distance of the applicants' place of residence from the university**.

Although this is never mentioned officially, during the informal conversation you learned that **those coming from too far away have a smaller chance of getting in**, partly for logistical reasons and partly because of the risk of student dropout.

What's more, your cartographer buddy did not let you down: he sent you a **function** with which you can **precisely compute the distance to the university** based on the applicants' county-level places of residence. This way you have the opportunity to build this information into your own prediction model as well, exactly the way the committee does.


In [ ]:
def haversine(lat1: float, lon1: float, lat2:float, lon2:float) -> float:
    """
    Computes the distance between two geographic points in kilometers using the Haversine formula.

    The Haversine formula is a calculation based on spherical trigonometry, which gives the
    great-circle distance between two geographic coordinates (latitude and longitude).

    Parameters:
        lat1 (float): The latitude of the first point (in degrees).
        lon1 (float): The longitude of the first point (in degrees).
        lat2 (float): The latitude of the second point (in degrees).
        lon2 (float): The longitude of the second point (in degrees).

    Return value:
        float: The distance between the two points in kilometers.

    Example:
        >>> haversine(47.4979, 19.0402, 48.8566, 2.3522)
        1244.51  # The distance between Budapest and Paris in kilometers (approximately)
    """
    R = 6371  # The radius of the Earth in kilometers
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2.0)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2.0)**2
    return 2 * R * np.arcsin(np.sqrt(a))

## **1.5 Training and Test Data**

In [ ]:
train_df = pd.read_csv('/kaggle/input/magyar-mi-diakolimpia-online-valogato-2025/train.csv')
test_df = pd.read_csv('/kaggle/input/magyar-mi-diakolimpia-online-valogato-2025/test.csv')

## **1.6 Column Descriptions**

- `Életkor` *(int)* – The applicant's age in years. Possible values: 17, 18 or 19.
- `Nem` *(int)* – The applicant's gender: 0 - male, 1 - female.
- `Osztályzat_9` *(float)* – The 9th-grade grade point average between 2.00 and 5.00, in steps of 0.25.
- `Osztályzat_10` *(float)* – The 10th-grade grade point average between 2.00 and 5.00, in steps of 0.25.
- `Osztályzat_11` *(float)* – The 11th-grade grade point average between 2.00 and 5.00, in steps of 0.25.
- `Osztályzat_12` *(float)* – The 12th-grade grade point average between 2.00 and 5.00, in steps of 0.25.
- `Történelem` *(int)* – History final-exam score (0–100).
- `Matematika` *(int)* – Mathematics final-exam score (0–100).
- `Magyar Nyelv és Irodalom` *(int)* – Hungarian language and literature final-exam score (0–100).
- `Informatika` *(int)* – Computer science final-exam score (0–100), if chosen; otherwise -1.
- `Biológia` *(int)* – Biology final-exam score (0–100), if chosen; otherwise -1.
- `Fizika` *(int)* – Physics final-exam score (0–100), if chosen; otherwise -1.
- `Angol` *(int)* – English final-exam score (0–100), if chosen; otherwise -1.
- `Német` *(int)* – German final-exam score (0–100), if chosen; otherwise -1.
- `Informatika_emelt` *(int)* – 1 if the applicant took the advanced-level final exam in computer science; 0 if standard-level; -1 if they did not choose the subject.
- `Biológia_emelt` *(int)* – 1 if the applicant took the advanced-level final exam in biology; 0 if standard-level; -1 if they did not choose it.
- `Fizika_emelt` *(int)* – 1 if the applicant took the advanced-level final exam in physics; 0 if standard-level; -1 if they did not choose it.
- `Angol_emelt` *(int)* – 1 if the applicant took the advanced-level final exam in English; 0 if standard-level; -1 if they did not choose it.
- `Német_emelt` *(int)* – 1 if the applicant took the advanced-level final exam in German; 0 if standard-level; -1 if they did not choose it.
- `Matematika_emelt` *(int)* – 1 if the score obtained in mathematics was earned at advanced level; 0 if at standard level.
- `Történelem_emelt` *(int)* – 1 if the applicant took the advanced-level final exam in history; 0 if standard-level.
- `Magyar Nyelv és Irodalom_emelt` *(int)* – 1 if the applicant took the advanced-level final exam in Hungarian; 0 if standard-level.
- `Szülői Végzettség` *(int)* – The parents' highest level of education: 1 - at most 8 grades, 2 - vocational, 3 - secondary school-leaving exam, 4 - college, 5 - university.
- `Középiskola Presztízse` *(float)* – The prestige value of the student's secondary school between 1 and 10, where 10 is the most prestigious.
- `Extrakurrikuláris Tevékenységek` *(int)* – Whether they participated in clubs, sports or other activities: 1 - yes, 0 - no.
- `Tanulási Szokások` *(int)* – The quality of study habits from 1 to 8: 1–3 irregular, 4–5 average, 6–8 consistent, deliberate studying.
- `Munkatapasztalat` *(int)* – Whether they have prior work experience: 1 - yes, 0 - no.
- `Ajánlások Száma` *(int)* – The number of recommendations obtained by the student (1–6), e.g. recommendations from teachers or professional mentors.
- `Versenyeken Való Részvétel` *(int)* – Whether they took part in academic competitions: 1 - yes, 0 - no.
- `Vármegye` *(string)* – The applicant's place of residence in textual form (e.g. "Baranya", "Pest").
- `Stressz Szint` *(float)* – The applicant's subjective stress level between 0.0 and 1.0, where 1.0 is maximum stress.
- `Felvételi Eredmény` *(int)* – The **binary target variable**: 1 if the applicant was admitted to the university; 0 if not.

In [ ]:
admitted_count = train_df['Felvételi Eredmény'].sum()
print(f"Number of Admitted Students in the Training Dataset: {admitted_count}")
print(f'Number of Items in the Training Dataset: {len(train_df)}')

In [ ]:
train_df.head()

# **2. Example Solution**

## **2.1 Logistic Regression**

In [ ]:
X_train = train_df[['Középiskola Presztízse']]
y_train = train_df['Felvételi Eredmény']

model = LogisticRegression()
model.fit(X_train, y_train)

In [ ]:
X_test = test_df[['Középiskola Presztízse']]

y_pred_test = model.predict(X_test)
y_pred_proba_test = model.predict_proba(X_test)[:, 1]

## **2.2 Exporting the Predictions in the Correct Format**

In [ ]:
test_idx = test_df['ID']
test_output_df = pd.DataFrame({'ID': test_idx, 'Felvételi Eredmény': y_pred_proba_test})
test_output_df.to_csv('HUN123_1.csv', index=False)

In [ ]:
test_output_df.head()

# **Uploading Predictions – Submission Guide**

If you decide to upload the `.csv` file containing your predictions, please pay attention to the following formats and naming rules:

### **Example CSV Format**

```csv
ID,Felvételi Eredmény
20079,0.405314
20080,0.401229
20081,0.401746
20082,0.402005
20083,0.397336
...
```

### **File naming convention**

- The file name should use the following format: *\<received_id>_\<checkpoint_number>.csv*:
  - `<received_id>` is the unique identifier you receive, which we send out to everyone 20 minutes before the online competition (e.g. `HUN123`)
  - `<checkpoint_number>` is the sequence number of the given attempt or upload (e.g. `1`, `2`, etc.)

**Example**: `HUN123_1.csv`

- Name the notebook file belonging to the prediction **using the same convention** and save it in `.ipynb` format:

**Example**: `HUN123_1.ipynb`

### **Saving files**

- Save the `.ipynb` notebook file **locally on your computer**.
- This is necessary because you must be able to submit the notebook belonging to your selected upload later as well (see the next point).

### **Finalizing your submission**

- **When registering on Kaggle, use the same e-mail address that you provided when registering for the competition (using Gmail is recommended).**  
- **You must use this same e-mail address to fill out the Google Form as well**, so that uploads can be unambiguously identified. (If using a Google account is not possible, you can also send your final solutions to the **mi_olimpia@inf.elte.hu** e-mail address.  
Please include your full name and your Kaggle username in the e-mail so that uploads can be unambiguously identified.)
- **On the Kaggle platform, your username must be your own name** for the duration of the competition, since **we cannot accept uploads made under pseudonyms or aliases**.
- **Within 10 minutes after the competition closes**, you must also submit the notebook belonging to your selected upload via a **Google Form**. Fill out the form only once!
- Verification and the calculation of the private score are based on the selected upload and the submitted notebook.

### **Multiple uploads**

- If you make several attempts, use increasing checkpoint numbers (e.g. `HUN123_2.csv`, `HUN123_3.csv`, etc.)
- Every `.csv` file must have its own saved `.ipynb` file with the same name.

---

**Don't forget:** the notebook and the prediction file are valid *as a pair*, so always be careful with exact file naming and saving!

**Google Forms:** [Link](https://forms.gle/Fsjjpiky1s72C7ta8)
